# Journal 3 - Fine-tuning on Vertex AI

**Question.** Does a small LLM fine-tuned with LoRA beat B2, and at what training cost?
**Answer.** Yes, by a wide margin: on val, **0.7 silent errors per 100 tickets against 28.3 for B2**, and no urgent ticket in a wrong desk (B2: 7.6 %), for 2 h 06 of a T4 GPU. The 3 outputs rejected by the contract are malformed ids copied from the ticket; they go to a human.

Model: Qwen2.5-3B-Instruct (better in French than the course's Phi-3-mini), LoRA on a 4-bit base, run as a Kubeflow pipeline. Course settings as a start: 3 epochs, rank 8, alpha 16, dropout 0.05, learning rate 3e-4.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks")]
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import _viz as viz
import _tracking
viz.setup()

from src import scoring
from src.baselines import predict_majority, predict_rules, fit_tfidf, predict_tfidf

## 1. Three decisions taken before the full run

| Check (free, local) | Measured | Decision |
|---|---|---|
| Do chats fit the course's 512 tokens? | median 495, p99 816; **790 of 1 922 (41 %) are longer** | sequences of 1 024 tokens: the JSON answer comes last and would be cut for 4 tickets in 10 |
| What does the loss learn from? | answer = ~30-50 tokens of ~500 | loss on the answer only, no packing |
| Is inference prompt = start of training text? | yes (checked on 3 examples) | same input in training and in use |

## 2. Two smoke runs (200 tickets, 40 steps) to calibrate

| Compute | Time | Loss at step 40 | Valid JSON on 20 val tickets |
|---|---|---|---|
| float32 (course default) | 654 s (16.4 s / step) | 0.050 | 0.90 |
| **float16** | **213 s (5.3 s / step)** | 0.049 | 0.95 |

float16 is **3.1x faster** with the same losses at every step and no NaN. Full run in float32 would have taken ~6 h; in float16, ~2 h. Decision: float16.

## 3. The full run: Qwen2.5-3B + LoRA, 3 epochs
What to read first: the loss curve (a rising validation loss means overfitting), then the val predictions scored with the protocol. Expected: fewer silent errors than B2, and no more urgent tickets in a wrong desk.

In [ ]:
history_path = ROOT / "docs/results/d_qwen2.5-3b_full_run_log_history.json"
hist = pd.DataFrame(json.loads(_tracking.load_training_history(ROOT, history_path)))
train = hist.dropna(subset=["loss"]).set_index("step")["loss"]
ev = hist.dropna(subset=["eval_loss"]).set_index("step")["eval_loss"]
runtime = hist["train_runtime"].dropna().iloc[0]
best = ev.idxmin()

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(train.index, train.values, color=viz.NEUTRAL, lw=1.4, label="train loss")
ax.plot(ev.index, ev.values, color=viz.ACCENT, lw=2, marker="o", ms=3.5, label="validation loss")
ax.scatter([best], [ev[best]], s=70, color=viz.ALERT, zorder=4)
ax.annotate(f"best: {ev[best]:.4f}\nstep {best}", (best, ev[best]), (best - 330, ev[best] * 5), fontsize=9,
            arrowprops=dict(arrowstyle="-", color=viz.MUTED))
for s in (432, 864):
    ax.axvline(s, color=viz.GRID, lw=1.2, zorder=0)
ax.set_yscale("log"); ax.set_yticks([1e-4, 1e-3, 1e-2, 1e-1], ["0.0001", "0.001", "0.01", "0.1"]); ax.minorticks_off(); ax.set_xlabel("step (grey lines: end of epochs 1 and 2)"); ax.set_ylabel("loss")
ax.legend(loc="upper right")
viz.headline(fig, "Training and validation loss", "Qwen2.5-3B + LoRA · optimizer step · T4 GPU")
viz.save(fig, "03_loss_curve")
plt.show()

Validation loss falls fast (0.045 at step 50), oscillates through epoch 2 (a peak at step 950 was noise, not overfitting), and reaches its minimum **0.0030 at the last evaluation, step 1250**. Nothing to cut. This says the model fits the answers; the scoring tells how well it reads.

## 4. Scoring the 293 val predictions against the baselines

In [ ]:
val_x, val_y, _ = scoring.load_split(ROOT / "data/dataset/val.csv")
train_x, train_y, _ = scoring.load_split(ROOT / "data/dataset/train.csv")
raw_d = _tracking.load_predictions(ROOT, "D", "val", ROOT / "docs/results/d_qwen2.5-3b_full_run_val_predictions.csv", val_x)
if raw_d is None:
    raise FileNotFoundError("No D/val predictions in MLflow or docs/results")
SHOWN = ["silent_error_rate", "needless_triage_rate", "urgent_wrong_desk", "auto_routed_rate", "desk_accuracy",
         "exact_transaction_id", "exact_amount", "recall_high_urgency", "severity_score", "valid_json_rate"]
val = {"B1": scoring.score(val_y, predict_rules(val_x)),
       "B2": scoring.score(val_y, predict_tfidf(fit_tfidf(train_x, train_y, 10.0), val_x)),
       "D": scoring.score(val_y, raw_d)}
display(pd.DataFrame(val).loc[SHOWN].round(3))

fig, ax = plt.subplots(figsize=(7.5, 2.6))
keys = ["B1", "B2", "D"]
viz.hbar(ax, ["B1 rules", "B2 TF-IDF + LR", "D fine-tuned"], [100 * val[k]["silent_error_rate"] for k in keys],
         [viz.SYSTEM_COLORS[k] for k in keys], fmt="{:.1f}")
ax.set_xlim(0, 85); ax.set_xlabel("silent errors per 100 tickets (lower is better)")
viz.headline(fig, "Silent errors on validation", "val • 293 tickets • selection view")
viz.save(fig, "03_val_silent")
plt.show()
print("valid JSON:", round(val["D"]["valid_json_rate"] * len(val_y)), "of", len(val_y))

D wins everywhere: desk accuracy 0.986 (B2 0.949), no urgent ticket in a wrong desk (B2 0.076), and the biggest gap is on references: exact id 0.986 and amount 0.990, against 0.894 and 0.829 for the regex-based baselines. It hands 0.7 ticket in 100 to a human without need, as B2 does, and routes 83 % of tickets by itself.

Two cautions: val has the same writer as train (the tests, written by another model, are the real check), and 293 tickets is small: 0.7 per 100 is 2 tickets.

## 5. The three outputs rejected by the contract

In [ ]:
FIELDS = ("intent", "sub_intent", "urgency", "sentiment", "transaction_id")
for i, r in enumerate(raw_d):
    if scoring.parse_output(r) is None:
        print(f"ticket {i}: model wrote id {json.loads(r)['transaction_id']!r} | label says {val_y[i].transaction_id!r}")
errors = dict.fromkeys(FIELDS, 0)
for truth, r in zip(val_y, raw_d):
    pred = scoring.parse_output(r)
    if pred:
        for f in FIELDS:
            errors[f] += getattr(pred, f) != getattr(truth, f)
print("errors among the valid outputs:", errors)

### Field errors on validation


In [ ]:
fields = ('intent','sub_intent','urgency','sentiment','transaction_id','amount','currency')
counts = {f: sum(scoring.parse_output(raw) is None or getattr(scoring.parse_output(raw), f) != getattr(truth, f)
                 for truth, raw in zip(val_y, raw_d)) for f in fields}
fig, ax = plt.subplots(figsize=(8, 3.7))
pd.Series(counts).sort_values().plot.barh(ax=ax, color=viz.ACCENT)
ax.set_xlabel('Incorrect or invalid outputs / 293 tickets'); ax.set_ylabel('Label field')
viz.headline(fig, 'D field errors', 'val · 293 tickets · invalid output counted once per field')
viz.save(fig, '03_field_errors')
plt.show()

The three rejected outputs are valid JSON: the model **copied a malformed id** (`TX-6084`, 4-5 digits instead of 6) that the contract refuses. They go to a human, which is the safe failure. This is the `id_malformed` trap: the model did not fully learn to answer `UNDEFINED` for an id that only looks like one. The valid outputs have few errors (`sub_intent` 4, `intent` 2, urgency 2, sentiment 1, id 1).

D is far better than B2 on val on both criteria (silent errors, urgent in a wrong desk). Next: shrink the model to run on a CPU.